In [2]:
from pathlib import Path

from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_mistralai import MistralAIEmbeddings
from langchain_community.vectorstores import FAISS

load_dotenv()

PDF_PATH = Path("../documents/hr_policy.pdf")

loader = PyPDFLoader(str(PDF_PATH))
documents = loader.load()

/var/folders/f_/1j85lpk92ddfd4c7r8d8rz880000gn/T/ipykernel_5403/1028077477.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = text_splitter.split_documents(documents)

embeddings = MistralAIEmbeddings(
    model="mistral-embed"
)

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings,
)

print(f"vector_store length: {vector_store.index.ntotal}")

vector_store length: 70


In [22]:
# query = "How many paid leaves can an employee take?"
query = "What are the working hours average per month"

# results = vector_store.similarity_search(
#     query,
#     k=1,
# )

results = vector_store.similarity_search_with_score(
    query,
    k=3,
)

print(f"Retrieved documents: {len(results)}")
for i, (doc,score) in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print(f"score: ${score}")
    print(doc.page_content)
    # print(doc.metadata)

Retrieved documents: 3

--- Result 1 ---
score: $0.3818531036376953
 All employees need to attain considering 8:40 working hours Average per Month.
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
This document contains trade secrets and proprietary information of Innvonix Tech Solutions Private Limited. Any Use, Disclos ure,  dissemination,  forwarding, 
printing or copying of this publication is absolutely prohibited without the express written permission of Innvonix Tech Solu tions Private Limited ©2023.If you are

--- Result 2 ---
score: $0.38376766443252563
deducted only for the working days. 
17. All the Saturdays of the month are off except Last Saturday of month (Training and development 
Saturdays) which will be counted as Working Day and employee has to attend minimum 7 hours to  
count it as a full day. However, the working hours of the last Saturday will not be counted in Monthly  
working hour’s average. 
18. Leaves application should

## Observation

Similarity search converts the user's query into an embedding and searches
the FAISS vector store for the most relevant document chunks.

The `k` parameter controls how many chunks are retrieved. A smaller `k` can
miss relevant information, while a larger `k` can introduce irrelevant
context.

`similarity_search_with_score()` also returns a score/distance for each
result. The meaning of the score depends on the similarity metric used by
the underlying vector index, so the score should not automatically be
interpreted as "higher is better".

Similarity search is part of the retrieval stage of RAG. The LLM has not
yet been involved.